# Binary Classification拡張モデル：学習からq-batch候補提案まで

Phase 2〜8のモデルを、連続・Mixed・階層構造ごとに比較します。
ここでは小さな合成データを使い、各モデルの`fit()`、`predict_proba()`と
分類Active Learningによる候補提案まで確認します。

- [分類モデル一覧](../../docs/models/classification.md)
- [理論](../../docs/theory/24_classification.md)
- [利用ガイド](../../docs/guide/24_classification.md)

`qPredictiveEntropy`は候補の周辺entropyの平均です。
候補間の情報重複を扱う`qBALD`とは目的が異なります。

In [ ]:
import torch
from botorch.optim import optimize_acqf, optimize_acqf_mixed

from robotorchan.acquisition import qPredictiveEntropy
from robotorchan.models.classification import (
    BinarySingleTaskGPClassifier,
    ContextualBinaryGPClassifier,
    HierarchicalBinaryGPClassifier,
    InfiniteWidthBNNBinaryGPClassifier,
    MixedInfiniteWidthBNNBinaryGPClassifier,
    MixedMultiFidelityBinaryGPClassifier,
    MixedSpectralMixtureBinaryGPClassifier,
    MultiFidelityBinaryGPClassifier,
    SpectralMixtureBinaryGPClassifier,
)

torch.manual_seed(17)
dtype = torch.double

## 1. 合成データとモデル構築

構造列はそれぞれカテゴリ、忠実度、Group / SubgroupのIDです。
学習ラベルは必ず0/1を使います。

In [ ]:
x = torch.linspace(0.05, 0.95, 12, dtype=dtype)
cat = torch.tensor([0.0, 1.0] * 6, dtype=dtype)
fidelity = torch.tensor([0.0, 1.0] * 6, dtype=dtype)
subgroup = torch.tensor([0.0, 2.0, 1.0, 3.0] * 3, dtype=dtype)
y = (x + 0.15 * cat > 0.55).to(dtype=dtype)

X_cont = x[:, None]
X_mixed = torch.stack((x, cat), dim=-1)
X_fidelity = torch.stack((x, fidelity), dim=-1)
X_mixed_fidelity = torch.stack((x, cat, fidelity), dim=-1)
X_hierarchy = torch.stack((x, cat, subgroup), dim=-1)

models = {
    "baseline": BinarySingleTaskGPClassifier(X_cont, y, inducing_points=6),
    "multi-fidelity": MultiFidelityBinaryGPClassifier(
        X_fidelity, y, data_fidelities=[1], inducing_points=6
    ),
    "mixed multi-fidelity": MixedMultiFidelityBinaryGPClassifier(
        X_mixed_fidelity, y, cat_dims=[1], data_fidelities=[2], inducing_points=6
    ),
    "spectral mixture": SpectralMixtureBinaryGPClassifier(
        X_cont, y, num_mixtures=2, inducing_points=6
    ),
    "mixed spectral mixture": MixedSpectralMixtureBinaryGPClassifier(
        X_mixed, y, cat_dims=[1], num_mixtures=2, inducing_points=6
    ),
    "contextual": ContextualBinaryGPClassifier(
        X_mixed, y, context_feature=1, inducing_points=6
    ),
    "hierarchical": HierarchicalBinaryGPClassifier(
        X_hierarchy, y, group_feature=1, subgroup_feature=2, inducing_points=6
    ),
    "infinite-width NNGP": InfiniteWidthBNNBinaryGPClassifier(
        X_cont, y, depth=2, inducing_points=6
    ),
    "mixed infinite-width NNGP": MixedInfiniteWidthBNNBinaryGPClassifier(
        X_mixed, y, cat_dims=[1], depth=2, inducing_points=6
    ),
}
train_inputs = {
    "baseline": X_cont,
    "multi-fidelity": X_fidelity,
    "mixed multi-fidelity": X_mixed_fidelity,
    "spectral mixture": X_cont,
    "mixed spectral mixture": X_mixed,
    "contextual": X_mixed,
    "hierarchical": X_hierarchy,
    "infinite-width NNGP": X_cont,
    "mixed infinite-width NNGP": X_mixed,
}

## 2. Variational ELBOによる学習と予測

比較のため少ないepochを使います。精度比較の結論には使用しません。

In [ ]:
for name, model in models.items():
    model.fit(epochs=8, lr=0.03)
    with torch.no_grad():
        probabilities = model.predict_proba(train_inputs[name][:3])
    assert probabilities.shape == (3, 2)
    print(name, "P(y=1):", probabilities[:, 1].tolist())

## 3. q=2の候補提案

連続入力の例ではBoTorch標準の`optimize_acqf`を利用します。
Mixedではカテゴリ列を固定する`optimize_acqf_mixed`を利用します。

In [ ]:
continuous_model = models["infinite-width NNGP"]
acq = qPredictiveEntropy(continuous_model)
candidate, value = optimize_acqf(
    acq_function=acq,
    bounds=torch.tensor([[0.0], [1.0]], dtype=dtype),
    q=2,
    num_restarts=2,
    raw_samples=12,
    options={"maxiter": 15},
)
print("continuous q=2:", candidate.detach(), "score:", value.detach())

mixed_model = models["mixed infinite-width NNGP"]
mixed_acq = qPredictiveEntropy(mixed_model)
mixed_candidate, mixed_value = optimize_acqf_mixed(
    acq_function=mixed_acq,
    bounds=torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype),
    q=2,
    num_restarts=2,
    raw_samples=12,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
    options={"maxiter": 15},
)
print("mixed q=2:", mixed_candidate.detach(), "score:", mixed_value.detach())

## 4. 利用上の注意

- `posterior()`は潜在GPの分布であり、クラス確率ではありません。
- Multi-Fidelityでは候補提案時に評価忠実度を固定するか、コストを考慮してください。
- Group / SubgroupのIDは非負整数で、Subgroupは一意のGroupに属します。
- `qPredictiveEntropy`は周辺スコア平均で、重複候補の抑制を保証しません。
- 本NotebookはAPIの利用例です。定量比較には独立したテストデータと複数seedが必要です。
- より詳しい比較は`benchmarks/binary_classification_extensions.py`を参照してください。

## 初学者向け補足：拡張モデルの選び方

最初からすべてのモデルを使う必要はありません。まず標準の二値GP分類で `fit()`、予測確率、候補提案の流れを理解し、入力やデータの構造に応じてモデルを選びます。

| データの特徴 | 検討するモデル・設定 | 注意点 |
|---|---|---|
| 一般的な連続入力 | 標準二値GP分類 | 最初の比較基準にする |
| 連続＋カテゴリ | Mixed分類 | カテゴリ列の符号化と候補固定を確認 |
| 複数の関連タスク | MultiTask分類 | タスクIDと特徴量を混同しない |
| 低／高忠実度の測定 | Multi-Fidelity分類 | 忠実度と実験コストを区別する |
| Group / Subgroup構造 | 階層型モデル | グループIDの意味を確認する |

### q=2の意味

`q=2` は **次に評価する候補を2点提案する** ことです。2クラス分類という意味ではありません。獲得関数と最適化器の組み合わせにより、2点を同時に最適化する場合と逐次的に選ぶ場合があります。このNotebookの各セルで使用する方法を確認してください。

### 実行時の確認事項

- 入力テンソルの各列が何を表すか、学習ラベルが0/1かを確認する。
- `fit()` の呼び出しと、予測確率を取得するAPIを区別する。
- 提案候補が変数範囲・カテゴリ・忠実度の条件を満たすか確認する。
- 少数epochの実行例はAPIのデモであり、モデル間の性能ランキングではない。

**学習の順番：** 39番（基本とActive Learning）→40番（ノイズ・入力の揺らぎ）→41番（確率の信頼性）→47番（拡張モデルとq-batch）を推奨します。